In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
 
df = spark.createDataFrame(data, columns)

# Write to Bronze Layer
df.write.format("delta").mode("append").save("/Volumes/workspace/default/medallion_architechture/bronze")


SILVER

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

df = spark.read.format("delta").load("/Volumes/workspace/default/medallion_architechture/bronze")

In [0]:
df = df.fillna({
    "amount": "0",
    "city": "Unknown"
})

In [0]:
from pyspark.sql.functions import to_date

df = df.withColumn("amount", col("amount").cast("int")) \
       .withColumn("date", to_date(col("date")))

In [0]:
df = df.filter(col("amount") > 0)

In [0]:
window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df = df.withColumn("rank", row_number().over(window_spec)) \
       .filter(col("rank") == 1) \
       .drop("rank")

In [0]:
df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/medallion_architechture/silver")

GOLD

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/medallion_architechture/silver")

Aggregations

In [0]:
# Total Sales by Product
from pyspark.sql.functions import col, row_number
sales_product = df.groupBy("product").agg(sum(col("amount")).alias("total_sales"))
sales_product.show()

+----------+-----------+
|   product|total_sales|
+----------+-----------+
|    Tablet|      22000|
|    Mobile|      33000|
|     Watch|       8000|
|    Laptop|     105000|
|Headphones|       3000|
+----------+-----------+



In [0]:
# Total sales by category
sales_category=df.groupBy("category").agg(sum(col("amount")).alias("total_sales"))
sales_category.show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     160000|
|Accessories|      11000|
+-----------+-----------+



In [0]:
# Total sales by city
sales_city=df.groupBy("city").agg(sum(col("amount")).alias("total_sales"))
sales_city.show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|    Delhi|      55000|
|  Chennai|      33000|
|  Unknown|       3000|
|Hyderabad|      72000|
|   Mumbai|       8000|
+---------+-----------+



Customer Metrics

In [0]:

# Number of orders per customer
orders_customer=df.groupBy("customer_id").count()
orders_customer.show()


+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C005|    1|
|       C004|    1|
|       C003|    1|
|       C001|    2|
|       C002|    1|
|       C007|    1|
+-----------+-----+



In [0]:
# Top customers based on spending
spending_customers=df.groupBy("customer_id").agg(sum("amount").alias( "total_spent"))
spending_customers.display()

customer_id,total_spent
C005,3000
C004,8000
C003,55000
C001,72000
C002,18000
C007,15000


TOP ANALYSIS

In [0]:

# Top Selling Product
top_product = sales_product.orderBy(col("total_sales").desc()).limit(1)
top_product.show()



+-------+-----------+
|product|total_sales|
+-------+-----------+
| Laptop|     105000|
+-------+-----------+



In [0]:
# Top customer
top_customer = spending_customers.orderBy(col("total_spent").desc()).limit(1)
top_customer.show()


+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|       C001|      72000|
+-----------+-----------+



In [0]:
sales_product.write.format("delta").mode("overwrite").save( "/Volumes/workspace/default/medallion_architechture/gold/product_sales")
sales_category.write.format("delta").mode("overwrite").save( "/Volumes/workspace/default/medallion_architechture/gold/category_sales")
sales_city.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/medallion_architechture/gold/city_sales")
spending_customers.write.format("delta").mode("overwrite").save( "/Volumes/workspace/default/medallion_architechture/gold/customer_spend")
orders_customer.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/medallion_architechture/gold/customer_orders")
top_product.write.format("delta").mode("overwrite").save( "/Volumes/workspace/default/medallion_architechture/gold/top_product")
top_customer.write.format("delta").mode("overwrite").save( "/Volumes/workspace/default/medallion_architechture/gold/top_customer")